# Scaled Dot-Product Attention

## 学习目标

能够从 Q、K、V 推导注意力分数，应用 padding/causal mask，并验证权重归一化。


## 概念模型与执行路径

Attention 用 query 与 key 的相似度决定如何汇总 value。除以 sqrt(d_k) 可控制高维点积方差；mask 在 softmax 前把不可见位置设为负无穷。


### 实验 1：定位 Attention 组件

**实验目的**：定位课程根目录，以导入最小 scaled dot-product attention 实现和终端演示。路径搜索只处理 notebook 启动目录差异。

In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 2：计算 scaled dot-product attention

**实验目的**：从 Q、K、V 的 shape 推导权重和输出。query/key 是 `(2,4,8)`，点积 `Q @ Kᵀ` 得到 `(2,4,4)`：每个 batch 中 4 个 query 对 4 个 key 的分数。除以 $\sqrt8$ 后沿最后一维 softmax，每个 query 行权重和为 1。

value 是 `(2,4,6)`，权重乘 V 后输出 `(2,4,6)`。输出最后一维来自 value，不需要等于 query/key 的特征维；但 key token 数必须与 value token 数一致。

**机制**：缩放控制高维随机点积的方差，避免 softmax 过早饱和、梯度变小。这里无可训练投影层，实验只展示 attention 核心运算。

In [ ]:
import torch
from common.models import ScaledDotProductAttention
torch.manual_seed(42)
query = torch.randn(2, 4, 8)
key = torch.randn(2, 4, 8)
value = torch.randn(2, 4, 6)
output, weights = ScaledDotProductAttention()(query, key, value)
print("scores -> weights:", weights.shape, "output:", output.shape)
print("row sums:", weights.sum(dim=-1))


### 实验 3：应用 causal mask 阻止关注未来

**实验目的**：构造下三角布尔 mask，使位置 i 只能查看位置 `≤i` 的 key。mask shape `(1,4,4)` 通过广播应用到两个 batch；课程实现约定 `True=可见`，不可见分数在 softmax 前填为 `-inf`。

因此严格上三角权重应为 0，future attention mass 应为 `0.0`。第一行只有 key 0 可见，所以其权重为 `[1,0,0,0]`。mask 必须在 softmax 前应用；softmax 后再置零会破坏行和为 1，除非重新归一化。

**边界**：若某个 query 的所有 key 都被 mask，softmax 面对全 `-inf` 可能产生 NaN。构造组合 mask 时要保证每个有效 query 至少有一个可见 key，或显式处理全遮蔽行。

In [ ]:
causal_mask = torch.ones(4, 4, dtype=torch.bool).tril().unsqueeze(0)
masked_output, masked_weights = ScaledDotProductAttention()(query, key, value, causal_mask)
print(masked_weights[0])
print("future attention mass:", masked_weights.triu(diagonal=1).sum().item())


### 实验 4：使用 PyTorch 多头自注意力

**实验目的**：把同一 tokens 同时作为 Q、K、V，执行两头 self-attention。`embed_dim=8,num_heads=2` 意味着每个头的维度是 4，必须满足 embed_dim 能被头数整除。

输出保持 `(2,4,8)`；默认返回的 `multi_weights` 已在 head 维求平均，因此 shape 为 `(2,4,4)`，不是 `(2,2,4,4)`。若要查看各头，传 `average_attn_weights=False`。模块内部包含 Q/K/V 投影与输出投影，这些参数是实验 2 的最小实现所没有的。

**mask API 注意**：`nn.MultiheadAttention` 的布尔 `attn_mask` 约定与课程实现不同——其中 `True` 表示不允许关注。迁移 mask 时不要直接复用方向。

In [ ]:
from torch import nn
multihead = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True)
tokens = torch.randn(2, 4, 8)
multi_output, multi_weights = multihead(tokens, tokens, tokens, need_weights=True)
print(multi_output.shape, multi_weights.shape)


### 实验 5：运行命令行 causal attention 冒烟测试

**实验目的**：运行与实验 2、3 相同的核心机制，并根据设备参数选择 CPU/CUDA/MPS。quick 模式使用 4 个 token，脚本打印输出 shape、每行权重和及未来注意力总量。

预期输出 shape 为 `(2,4,16)`，权重行和接近 1，future attention mass 为 0。这个脚本没有训练，也没有多头投影；它是数值与 mask 契约的可执行检查。

In [ ]:
# python 07-deep-learning/pytorch/examples/attention_demo.py --quick


## 底层机制

多头注意力先把 embedding 投影到多个子空间，各头独立计算 attention，再拼接并做输出投影。不同头可以学习不同关系，但并不保证自然形成可解释语义。计算和显存的主要瓶颈是 `(query_tokens,key_tokens)` 分数矩阵，self-attention 对序列长度通常是平方复杂度。

padding mask 屏蔽不存在的 key，causal mask 阻止看到未来；两者可能需要广播后组合。Attention 权重描述模型当前计算中的加权路径，但不能直接当作可靠的因果解释。

## 检查点

回答并验证：1）为什么权重是 `(2,4,4)`、输出是 `(2,4,6)`？2）softmax 应沿哪一维？3）除以 `sqrt(d_k)` 的作用是什么？4）第一行 causal 权重为何是 `[1,0,0,0]`？5）全 mask 行有什么风险？6）多头权重默认为何没有 head 维？7）课程 mask 与 MultiheadAttention 布尔 mask 的 True 语义有何不同？

## 试一试

构造 padding mask 让最后两个 key 不可见，验证相应列为 0 且行和仍为 1；再与 causal mask 组合。设置 `average_attn_weights=False` 查看每个头，比较不同 head。最后改变 query token 数与 key/value token 数，构造 cross-attention 并推导 shape。

## 常见错误与调试

- **mask True/False 语义混淆**：不同 API 约定相反；先读契约并做小矩阵检查。
- **softmax 维度错误**：应对每个 query 的 key 维归一化；检查行和。
- **忘记缩放**：高维点积使 softmax 饱和；除以 `sqrt(d_k)`。
- **key/value 长度不一致**：权重无法乘 V；确保共享 key-token 维。
- **全 mask 行**：可能产生 NaN；保证至少一个可见 key 或特殊处理。
- **误读多头权重 shape**：默认已平均 heads；需要时关闭平均。
- **把权重当可靠解释**：它不是因果归因；结合扰动和其他分析。